# M3 plate_v2a 训练（Colab T4）
前置：在 Windows 本机跑完 `build_dataset.py`，把 `datasets/plate_v2/` 打包成 `plate_v2.zip`（含 data.yaml + train/val + dataset_card.json）。
全程约 1–2 小时（T4，150 epochs）。Kaggle P100 同理（每周 30h 免费）。
路线细节：`docs/M3_TRAINING_PLAN.md`。

In [ ]:
BRANCH = "arena/01a08ef3-vbt-research"  # 合并到 main 后改 "main"
DATA_ZIP = "/content/plate_v2.zip"  # 或 Drive 路径 /content/drive/MyDrive/plate_v2.zip
NAME = "plate_v2a"
!nvidia-smi -L

In [ ]:
!git clone --depth 1 --branch $BRANCH https://github.com/DanielHo01/VBT-Research.git /content/vbt 2>&1 | tail -1
!pip install -q -r /content/vbt/requirements-train.txt 2>&1 | tail -1
import ultralytics; print("ultralytics", ultralytics.__version__)

## 上传数据（二选一）
zip 约 1–1.5GB：Drive 挂载最稳（右侧文件→挂载）；`files.upload()` 备选（大文件易断）。

In [ ]:
# 方案 A：Drive（推荐）
from google.colab import drive
drive.mount('/content/drive')
# DATA_ZIP = "/content/drive/MyDrive/plate_v2.zip"

# 方案 B：直接上传（取消注释用）
# from google.colab import files
# files.upload()  # 选择 plate_v2.zip

In [ ]:
!unzip -q "$DATA_ZIP" -d /content/plate_v2_data && find /content/plate_v2_data -maxdepth 3 -name data.yaml
!ls /content/plate_v2_data

## 训练（~1–2h）
若上一步 data.yaml 不在 `/content/plate_v2_data/plate_v2/`，改下一格的 `--data` 路径。

In [ ]:
!cd /content/vbt && python scripts/train_plate_v2.py --data /content/plate_v2_data/plate_v2/data.yaml --name $NAME --batch 32 --device 0

## 回传：下载 onnx + run_card，回 Windows 跑 Gate C

In [ ]:
from google.colab import files
import glob
onnx = glob.glob(f"/content/vbt/runs/plate_v2/{NAME}/weights/best.onnx")[0]
card = f"/content/vbt/runs/plate_v2/{NAME}/run_card.json"
print(onnx, card)
files.download(onnx)
files.download(card)

## 回传后（Windows 本机）
```bat
copy best.onnx models\plate_v2a.onnx
python scripts/run_benchmark_v0.py --tag plate_v2a --model models/plate_v2a.onnx --engine "vbtcore v1.5 + plate_v2a"
```
门：假拒绝 6→≤2 且无回归（见 M3 文档 §7）。过了再跑 holdout 双报。